# Coffee17 Master Representation Screening V1

Unified 5-fold seed-42 development screen. Anchors B0/B1 are trained once per fold, followed by WR-HBP, MFR, LRBP, adaptive multistage fusion, fusion loss, WCA, Fda, Fca, DWT dual-domain, FFC and GFNet-style global filtering. **Outer test is not accessed.**


In [ ]:
NOTEBOOK_BUILD = "MASTER-REPRESENTATION-SCREEN-V1"
NOTEBOOK_RUNTIME_REV = "SCREENING-COMPAT-V3-20261006"
SCIENTIFIC_CODE_COMMIT = "b1ac6ca548ebb335d69066377d6f1a1fbb56df36"

import importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-master-screen-v1"
PROJECT = WORK / "coffee17-master-representation-screening-v1-runtime-v3"
OUT = PROJECT / "experiments"
CONFIG_REL = "configs/representation_screening/MASTER_SCREENING_V1.yaml"
CANDIDATES = [
    "B2",
    "R1", "R2", "R3", "R4", "R5",
    "W1", "W2", "W3", "W4", "W5", "W6", "W7",
    "F1", "F2",
]

assert INPUT.is_dir() and WORK.is_dir(), "Notebook harus dijalankan di Kaggle."

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("NOTEBOOK_RUNTIME_REV:", NOTEBOOK_RUNTIME_REV)
print("SCIENTIFIC_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)
print("CANDIDATES:", CANDIDATES)

# Rerun safety: a previous execution may have left cwd inside REPO.
# Move out before deleting/recloning it, otherwise git fails with
# "Unable to read current working directory: No such file or directory".
os.chdir(WORK)
if REPO.exists():
    shutil.rmtree(REPO)
run(
    ["git", "clone", "--quiet", "https://github.com/ediprin/coffee-bean-classification.git", REPO],
    cwd=WORK,
)
run(["git", "-C", REPO, "checkout", "--quiet", "--detach", SCIENTIFIC_CODE_COMMIT])
run([sys.executable, "-m", "pip", "install", "-q", "-r", REPO / "requirements/preprocessing-study.txt"])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")
print("GPU:", torch.cuda.get_device_name(0))

# Fail fast before expensive training.
run([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_frequency_screening.py",
    "tests/test_representation_screening.py",
    "tests/test_master_screening.py",
], cwd=REPO)

# This broad architecture screen intentionally uses the same seeded development
# regime as the successful historical Coffee17 screens. Recovery-grade
# torch.use_deterministic_algorithms(True) is NOT enabled here because several
# valid CUDA operators (e.g. adaptive pooling in multi-stage fusion) have no
# deterministic backward implementation on T4/PyTorch.
print("DETERMINISM_MODE: SEEDED_MATCHED (torch strict determinism OFF)")

# GPU backward preflight: every arm must complete a real CUDA forward+backward
# before any expensive Coffee17 fold training starts.
from bilinear_lmmd.core.config import load_config
from bilinear_lmmd.experiments.run_master_screening_v1 import (
    MASTER_CANDIDATES,
    build_master_model,
    _forward,
)

_smoke_cfg = load_config(REPO / CONFIG_REL)
_smoke_cfg["device"] = "cuda:0"
_smoke_cfg["model"]["pretrained"] = False
_smoke_raw = torch.rand(2, 3, 224, 224, device="cuda:0")

print("\n=== CUDA FORWARD/BACKWARD PREFLIGHT ===")
for _candidate in MASTER_CANDIDATES:
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    _model = build_master_model(_candidate, _smoke_cfg).to("cuda:0").train()
    _out = _forward(_candidate, _model, _smoke_raw)
    if _out.logits.shape != (2, 17):
        raise RuntimeError(
            f"{_candidate}: logits shape salah: {tuple(_out.logits.shape)}"
        )
    _smoke_loss = _out.logits.float().square().mean()
    _smoke_loss.backward()
    if not any(
        p.grad is not None and torch.isfinite(p.grad).all()
        for p in _model.parameters()
        if p.requires_grad
    ):
        raise RuntimeError(f"{_candidate}: tidak ada finite trainable gradient.")
    print(f"CUDA PREFLIGHT PASS: {_candidate}")
    del _model, _out, _smoke_loss
    torch.cuda.empty_cache()
del _smoke_raw, _smoke_cfg
print("CUDA PREFLIGHT: ALL ARMS PASS")

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)

PROJECT.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)

# Optional resume: if a previous project tree is attached as a Kaggle input,
# copy only its experiments/logs into the current project.
for candidate_root in INPUT.rglob("coffee17-master-representation-screening-v1-runtime-v3"):
    if candidate_root.is_dir() and candidate_root != PROJECT:
        print("RESTORE PRIOR PROJECT:", candidate_root)
        prior_exp = candidate_root / "experiments"
        prior_logs = candidate_root / "logs"
        if prior_exp.is_dir():
            shutil.copytree(prior_exp, OUT, dirs_exist_ok=True)
        if prior_logs.is_dir():
            shutil.copytree(prior_logs, PROJECT / "logs", dirs_exist_ok=True)
        break

by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus Coffee17 original: "
        f"expected 979 images / 17 classes, observed {raw_count}/{len(by_class)}."
    )

ARCHIVE = WORK / "coffee17_master_screen_original.zip"
PROV = WORK / "coffee17_master_screen_provenance"
CANONICAL = WORK / "coffee17_master_screen_original_v1"
FOLDS = WORK / "coffee17_master_screen_folds"

for path in (PROV, CANONICAL, FOLDS):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE,
    PROV,
    canonical_root=CANONICAL,
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV / "coffee17_provenance.json",
    FOLDS,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

CONFIG = REPO / CONFIG_REL
for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_master_screen_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)
    materialize_preprocessing_development(
        CANONICAL,
        FOLDS / "clean_manifest.json",
        FOLDS / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_master_screening_v1",
        "--config", CONFIG,
        "--fold", str(fold),
        "--data-root", DEV,
        "--output-root", OUT,
        "--candidates", *CANDIDATES,
        "--device", "cuda:0",
        "--resume",
        "--authorize-training",
        "--required-commit", SCIENTIFIC_CODE_COMMIT,
    ], cwd=REPO, log_path=PROJECT / "logs" / f"fold_{fold}.log")

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

SUMMARY = PROJECT / "analysis" / "master_screening_v1.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_master_screening_summary_v1",
    "--output-root", OUT,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
print("\n=== MASTER RESULT ===")
ranking = sorted(
    (
        (
            candidate,
            summary["aggregate"][candidate]["delta_vs_anchor"]["macro_f1"]["mean"],
        )
        for candidate in summary["candidates"]
        if candidate not in {"B0", "B1"}
    ),
    key=lambda row: row[1],
    reverse=True,
)
for candidate, delta in ranking:
    item = summary["aggregate"][candidate]
    print(
        f"{candidate:>2} [{item['family']}] vs {item['anchor']}: "
        f"Macro={item['metrics']['macro_f1']['mean']:.4%} "
        f"dMacro={delta:+.4%} "
        f"positive={item['positive_delta_folds']['macro_f1']}/5 "
        f"dHard={item['delta_vs_anchor']['hard_class_f1']['mean']:+.4%} "
        f"dWorst={item['delta_vs_anchor']['worst_class_f1']['mean']:+.4%} "
        f"params={item['parameter_count']:,}"
    )

zip_path = Path(shutil.make_archive(
    str(WORK / "coffee17-master-representation-screening-v1"),
    "zip",
    root_dir=PROJECT,
))
print("\nREADY:", zip_path)
print("SIZE:", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")
print("Outer test was NOT accessed.")
